# Assignment 3 - PCD
Alif Ryanatha Azhar Suryanto - 25/569917/PA/24023

Tugas: segmentasi + operasi morfologi tanpa pakai library (jadi gaboleh pakai cv2.threshold, cv2.dilate, dll). Terus bandingin hasil segmentasi sebelum dan sesudah dikasih morfologi.

aku bikin gambar sendiri pakai numpy soalnya biar ga ribet upload2 file

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

### bikin gambar test (kotak + lingkaran + noise)

In [ ]:
size = 200
img = np.full((size, size), 40, dtype=np.uint8)

img[30:90, 30:110] = 200

yy, xx = np.ogrid[:size, :size]
circle = (xx-150)**2 + (yy-60)**2 <= 35**2
img[circle] = 210

blob = ((xx-100)/55)**2 + ((yy-150)/30)**2 <= 1
img[blob] = 190

img_clean = img.copy()

# noise salt and pepper manual
h, w = img.shape
n = int(0.06 * h * w)
for i in range(n//2):
    y, x = np.random.randint(0,h), np.random.randint(0,w)
    img[y,x] = 255
for i in range(n//2):
    y, x = np.random.randint(0,h), np.random.randint(0,w)
    img[y,x] = 0

plt.imshow(img, cmap='gray')
plt.title('test image + noise')
plt.show()

### segmentasi pake otsu

ini threshold otomatis, nyari T yang bikin variance antar kelas foreground-background paling gede. aku itung histogram manual terus looping tiap T dari 0-255.

In [ ]:
def otsu(img):
    hist = np.zeros(256)
    for v in img.ravel():
        hist[v] += 1

    total = img.size
    sum_all = sum(i*hist[i] for i in range(256))

    sum_bg = 0
    w_bg = 0
    best_T = 0
    best_var = 0

    for T in range(256):
        w_bg += hist[T]
        if w_bg == 0:
            continue
        w_fg = total - w_bg
        if w_fg == 0:
            break
        sum_bg += T*hist[T]
        m_bg = sum_bg/w_bg
        m_fg = (sum_all - sum_bg)/w_fg
        var_between = w_bg*w_fg*(m_bg-m_fg)**2
        if var_between > best_var:
            best_var = var_between
            best_T = T
    return best_T

T = otsu(img)
print("threshold:", T)

seg = np.zeros_like(img)
seg[img > T] = 1

plt.imshow(seg, cmap='gray')
plt.title(f'hasil segmentasi (T={T}) - belum di morfologi')
plt.show()

### morfologi manual

structuring elementnya 3x3 semua 1. dilation = kalo ada minimal 1 tetangga foreground jadi foreground. erosion = kalo semua tetangga foreground baru tetep foreground, selain itu jadi background. opening = erosi dulu baru dilasi. closing = dilasi dulu baru erosi.

In [ ]:
kernel = np.ones((3,3))

def dilate(bimg):
    padded = np.pad(bimg, 1)
    out = np.zeros_like(bimg)
    for i in range(bimg.shape[0]):
        for j in range(bimg.shape[1]):
            window = padded[i:i+3, j:j+3]
            if window.max() == 1:
                out[i,j] = 1
    return out

def erode(bimg):
    padded = np.pad(bimg, 1)
    out = np.zeros_like(bimg)
    for i in range(bimg.shape[0]):
        for j in range(bimg.shape[1]):
            window = padded[i:i+3, j:j+3]
            if window.min() == 1:
                out[i,j] = 1
    return out

def opening(bimg):
    return dilate(erode(bimg))

def closing(bimg):
    return erode(dilate(bimg))

d = dilate(seg)
e = erode(seg)
o = opening(seg)
c = closing(seg)

fig, ax = plt.subplots(2,2, figsize=(8,8))
ax[0,0].imshow(d, cmap='gray'); ax[0,0].set_title('dilation')
ax[0,1].imshow(e, cmap='gray'); ax[0,1].set_title('erosion')
ax[1,0].imshow(o, cmap='gray'); ax[1,0].set_title('opening')
ax[1,1].imshow(c, cmap='gray'); ax[1,1].set_title('closing')
for a in ax.ravel():
    a.axis('off')
plt.show()

### bandingin sebelum vs sesudah morfologi

noise salt pepper bikin ada titik2 putih doang di background sama lubang item kecil di dalem bentuk. jadi aku pake opening dulu (buang titik2 noise) terus closing (nutup lubang kecil).

In [ ]:
seg_clean = closing(opening(seg))

fig, ax = plt.subplots(1,3, figsize=(14,5))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('input')
ax[1].imshow(seg, cmap='gray'); ax[1].set_title('tanpa morfologi')
ax[2].imshow(seg_clean, cmap='gray'); ax[2].set_title('pake morfologi')
for a in ax:
    a.axis('off')
plt.show()

# cek akurasi ke ground truth (gambar tanpa noise)
T2 = otsu(img_clean)
gt = np.zeros_like(img_clean)
gt[img_clean > T2] = 1

acc_before = np.mean(seg == gt) * 100
acc_after = np.mean(seg_clean == gt) * 100

print("akurasi sebelum morfologi:", round(acc_before,2), "%")
print("akurasi sesudah morfologi:", round(acc_after,2), "%")
print("jumlah pixel foreground sebelum:", int(seg.sum()))
print("jumlah pixel foreground sesudah:", int(seg_clean.sum()))

### kesimpulan

tanpa morfologi hasil segmentasinya masih banyak noise (titik2 putih di background sama lubang2 kecil di dalem bentuk). setelah dikasih opening+closing noise nya ilang karena ukurannya lebih kecil dari structuring element yang dipake, sedangkan bentuk aslinya (kotak, lingkaran, blob) tetep ke-preserve karena ukurannya lebih gede. akurasi pixel dibanding ground truth juga naik setelah dikasih morfologi.

link github: (isi sendiri sebelum submit)